# DDPM Forward Process (Closed Form) -- No Neural Network

This notebook implements and visualizes **only** the DDPM forward
(noising) process -- there is no model, no training, no loss. The
forward process is a fixed Markov chain (Ho et al., 2020, Eq. 2):

$$q(x_t \mid x_{t-1}) = \mathcal{N}(x_t;\ \sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I)$$

Iterating this chain from $x_0$ gives a closed-form "jump straight to
step $t$" formula (paper Eq. 4, reparameterized with the Gaussian
sampling trick):

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

| symbol | code variable | shape | meaning |
|---|---|---|---|
| $x_0$ | `x0` | `[C,H,W]` | clean image, pixels in $[-1,1]$ (DDPM convention) |
| $t$ (0-indexed) | `t_indices` | `[K]` | which timesteps to sample $x_t$ at |
| $\bar\alpha_t$ | `alpha_bars[t_indices]` | `[K]` | cumulative signal-retention factor (`code/beta_schedule.py`) |
| $\sqrt{\bar\alpha_t}$ | `sqrt_alpha_bar_t` | `[K]` | **signal coefficient**: weight on $x_0$ |
| $\sqrt{1-\bar\alpha_t}$ | `sqrt_one_minus_alpha_bar_t` | `[K]` | **noise coefficient**: weight on $\varepsilon$ |
| $\varepsilon$ | `noise` | `[K,C,H,W]` | standard Gaussian noise |
| $x_t$ | `x_t` | `[K,C,H,W]` | noised image at each requested $t$ |

**Indexing convention**: identical to `beta_schedule.ipynb` -- `t_indices`
are plain 0-indexed positions into the length-$T$ `alpha_bars` array
(index $0 \leftrightarrow$ paper's $t{=}1$, index $T{-}1 \leftrightarrow$
paper's $t{=}T$). We visualize `t = [0, 100, 300, 400, 600, 800, 999]`
with $T=1000$.

**No dataset download**: $x_0$ is a small procedurally generated
synthetic image (gradient background + a solid circle, for clear edge
structure) -- this keeps the exercise entirely about the forward-
process math, with no model and no GPU required.

Core code lives in `code/ddpm_forward_process.py`
(`forward_diffusion_sample`, `get_signal_and_noise_coefficients`,
`signal_noise_table`, `visualize_forward_process`), which reuses
`code/beta_schedule.py` for $\bar\alpha_t$ rather than recomputing it;
both are unit-checked (`python code/ddpm_forward_process.py
--smoke-test`). This notebook is a thin orchestration layer.

## 1. Clone/pull repository

Forward-process code lives in `code/ddpm_forward_process.py` (built on `code/beta_schedule.py`), not duplicated in this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 2. Build the beta/alpha_bar schedule and the synthetic x0

Same `T=1000`, `beta_start=1e-4`, `beta_end=0.02` linear schedule as `beta_schedule.ipynb` (reused, not recomputed).

In [ ]:
import torch

from beta_schedule import linear_beta_schedule, compute_alpha_and_alpha_bar
from ddpm_forward_process import (
    make_synthetic_clean_image,
    forward_diffusion_sample,
    signal_noise_table,
    visualize_forward_process,
)

NUM_TIMESTEPS = 1000
betas = linear_beta_schedule(NUM_TIMESTEPS, beta_start=1e-4, beta_end=0.02)  # [T]
_alphas, alpha_bars = compute_alpha_and_alpha_bar(betas)                     # [T]

x0 = make_synthetic_clean_image(height=64, width=64)  # [3,64,64], pixels in [-1,1]
print(f"x0.shape = {tuple(x0.shape)}, range = [{x0.min().item():.3f}, {x0.max().item():.3f}]")


## 3. Sample x_t at t = [0, 100, 300, 400, 600, 800, 999]

One shared `x0`, one independent noise draw `eps` per requested `t` (fixed seed for reproducibility), applied via the single closed-form equation above -- no iterative stepping through the Markov chain needed.

In [ ]:
t_indices = torch.tensor([0, 100, 300, 400, 600, 800, 999])
generator = torch.Generator().manual_seed(0)

x_t, noise, sqrt_alpha_bar_t, sqrt_one_minus_alpha_bar_t = forward_diffusion_sample(
    x0, t_indices, alpha_bars, generator=generator
)
print(f"x_t.shape = {tuple(x_t.shape)}")


## 4. Signal and noise coefficients at each visualized t

$\sqrt{\bar\alpha_t}$ (signal coefficient, weight on $x_0$) and
$\sqrt{1-\bar\alpha_t}$ (noise coefficient, weight on $\varepsilon$),
printed for exactly the 7 timesteps visualized below. Note
$\sqrt{\bar\alpha_t}^2 + \sqrt{1-\bar\alpha_t}^2 = \bar\alpha_t + (1-\bar\alpha_t) = 1$
exactly, for every $t$ (checked in the smoke test) -- the two
coefficients always trade off against each other.

In [ ]:
print(signal_noise_table(t_indices, sqrt_alpha_bar_t, sqrt_one_minus_alpha_bar_t))


## 5. Visualize x0 and x_t at each requested t

Left to right: the clean $x_0$, then $x_t$ at increasing $t$. The
circle's sharp edge is the first structure to blur into noise; by
$t=300$–$400$ (signal coefficient $\approx 0.63 \to 0.44$) the
background gradient is already unrecognizable, and by $t=800$–$999$
(signal coefficient $\approx 0.04 \to 0.006$) the image is
visually indistinguishable from pure Gaussian noise -- consistent with
$\bar\alpha_t \to 0$ seen in `beta_schedule.ipynb`. Pixels are mapped
from $[-1,1]$ back to $[0,1]$ and clamped purely for display (`imshow`
needs a valid range); the underlying $x_t$ values are not clamped.

In [ ]:
visualize_forward_process(x0, x_t, t_indices, sqrt_alpha_bar_t, sqrt_one_minus_alpha_bar_t)
